# CodeGen — Group 45
## Step 2: The Data Engine — building validated Python→Rust pairs

**What we are building:** the training data for our project. We take simple Python problems,
translate each into Rust with a small coder model, and **keep only the Rust that compiles and
passes the tests** — using the exact harness from Step 1 as the filter. The output is
`pairs.jsonl`, a file of verified (Python → Rust) examples we will fine-tune on in Step 3.

**The key idea:** we don't *find* a Python→Rust corpus (one barely exists). We **generate and
validate** it — the MultiPL-T recipe. A noisy translator is fine, because the harness throws
away everything that doesn't pass.

**Leakage rule:** we build training data from **MBPP** problems and will **evaluate on
HumanEval-Rust** (Step 1) — two disjoint problem sets, so we never train on what we test.

**To run:** needs a **GPU** (`Runtime → Change runtime type → T4 GPU`). Then `Runtime → Run all`.


## 1. Install Rust + Python libraries
Note: we do **not** upgrade `torch` — Colab ships a matched `torch`/`torchvision` pair, and
force-upgrading torch breaks it (the `torchvision::nms` error). We only add transformers etc.

In [ ]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version
# IMPORTANT: do NOT add `torch` here — upgrading it breaks Colab's torchvision.
!pip install -q -U datasets transformers accelerate
print("setup done")


warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.96.0 (ac68faa20 2026-05-25)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.96.0 (ac68fa

## 2. The harness (same as Step 1 — our validator)
This is the unchanged `evaluate_one` from Step 1. Here it plays a new role: the **quality
filter** that decides which generated Rust is good enough to keep as training data.

In [ ]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")


harness ready


## 3. Load training problems (MBPP-Rust) + their Python solutions
- **mbpp-rs** (from MultiPL-E) gives us 354 Rust problems, each with a signature and **ready Rust
  unit tests** — perfect for validation.
- **MBPP** (the original Python dataset) gives us the **Python solution** for each problem.

We join them by task id (the `mbpp_<id>_...` in the Rust name matches MBPP's `task_id`).

In [ ]:
from datasets import load_dataset
import re

def load_rs(cfg):
    try:
        return load_dataset("nuprl/MultiPL-E", cfg, split="test")
    except Exception:
        return load_dataset("nuprl/MultiPL-E", cfg, split="test", trust_remote_code=True)

train_rs = load_rs("mbpp-rs")          # Rust problems + tests (our training source)

# Python solutions from MBPP, across all splits, keyed by task_id
mbpp = load_dataset("google-research-datasets/mbpp", "full")
py_by_id = {}
for split in mbpp:
    for ex in mbpp[split]:
        py_by_id[ex["task_id"]] = ex["code"]

def mbpp_id(name):
    m = re.match(r"mbpp_(\d+)_", name)
    return int(m.group(1)) if m else None

problems = []
for ex in train_rs:
    pid = mbpp_id(ex["name"])
    problems.append({"name": ex["name"], "prompt": ex["prompt"], "tests": ex["tests"],
                     "stop_tokens": ex["stop_tokens"], "task_id": pid,
                     "python": py_by_id.get(pid)})

have_py = sum(p["python"] is not None for p in problems)
print(len(problems), "Rust training problems;", have_py, "matched to a Python solution")
print("\n=== Example problem ===")
print("PYTHON:\n", problems[0]["python"])
print("RUST PROMPT:\n", problems[0]["prompt"])
print("RUST TESTS:\n", problems[0]["tests"][:200], "...")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

mbpp-rs/test-00000-of-00001.parquet:   0%|          | 0.00/72.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/354 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

354 Rust training problems; 354 matched to a Python solution

=== Example problem ===
PYTHON:
 import math
def is_not_prime(n):
    result = False
    for i in range(2,int(math.sqrt(n)) + 1):
        if n % i == 0:
            result = True
    return result
RUST PROMPT:
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {

RUST TESTS:
 }

fn main() {
    let candidate = is_not_prime;
    assert_eq!(candidate(2), false);
    assert_eq!(candidate(10), true);
    assert_eq!(candidate(35), true);
    assert_eq!(candidate(37), false);
}
 ...


## 4. Load the translator model
We use a small but capable coder model — **Qwen2.5-Coder-1.5B** — to do the Python→Rust
translation. It runs on a free T4. (we can swap `TRANSLATOR` for a bigger model or an API
later to raise the yield — the harness filter stays the same either way.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TRANSLATOR = "Qwen/Qwen2.5-Coder-1.5B"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
ttok = AutoTokenizer.from_pretrained(TRANSLATOR)
tmodel = AutoModelForCausalLM.from_pretrained(TRANSLATOR, torch_dtype=dtype)
tmodel = tmodel.to("cuda" if torch.cuda.is_available() else "cpu")
print("translator loaded on", tmodel.device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

translator loaded on cuda:0


## 5. The engine: translate --> validate --> keep
For each problem we build a prompt that shows the model the **Python as a reference comment**
and then the **Rust signature to complete** (this "parallel pairing" is what the scaling-laws
paper recommends). The model writes the Rust body; we trim it at the stop token; we run it
through the harness; and we **keep only the ones that pass**.

In [ ]:
import json

def python_as_comment(py):
    if not py:
        return ""
    body = "\n".join("// " + line for line in py.strip().splitlines())
    return "// Reference Python implementation:\n" + body + "\n"

def translate_body(problem, max_new_tokens=256):
    prompt = python_as_comment(problem["python"]) + problem["prompt"]
    inputs = ttok(prompt, return_tensors="pt").to(tmodel.device)
    out = tmodel.generate(**inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, pad_token_id=ttok.eos_token_id)
    text = ttok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    cut = len(text)                      # keep only the function body
    for s in problem["stop_tokens"]:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut]

def build_pairs(limit=40):
    pairs, attempted, passed = [], 0, 0
    for p in problems[:limit]:
        attempted += 1
        body = translate_body(p)
        status = evaluate_one(p["prompt"], body, p["tests"])
        if status == "pass":
            passed += 1
            rust_solution = p["prompt"] + body + "}"     # signature + body + closing brace
            pairs.append({"task": p["name"], "python": p["python"],
                          "rust_prompt": p["prompt"], "rust_solution": rust_solution})
        if attempted % 10 == 0:
            print(f"  {attempted} attempted, {passed} validated ({100*passed/attempted:.0f}% yield)")
    print(f"\nDONE: {passed}/{attempted} validated pairs kept ({100*passed/max(attempted,1):.0f}% yield)")
    return pairs

# Start small to confirm it works, then raise the limit (up to len(problems) = 354).
pairs = build_pairs(limit=354)


  10 attempted, 5 validated (50% yield)
  20 attempted, 12 validated (60% yield)
  30 attempted, 17 validated (57% yield)
  40 attempted, 25 validated (62% yield)
  50 attempted, 30 validated (60% yield)
  60 attempted, 35 validated (58% yield)
  70 attempted, 40 validated (57% yield)
  80 attempted, 47 validated (59% yield)
  90 attempted, 52 validated (58% yield)
  100 attempted, 58 validated (58% yield)
  110 attempted, 64 validated (58% yield)
  120 attempted, 71 validated (59% yield)
  130 attempted, 78 validated (60% yield)
  140 attempted, 83 validated (59% yield)
  150 attempted, 88 validated (59% yield)
  160 attempted, 96 validated (60% yield)
  170 attempted, 99 validated (58% yield)
  180 attempted, 104 validated (58% yield)
  190 attempted, 110 validated (58% yield)
  200 attempted, 117 validated (58% yield)
  210 attempted, 122 validated (58% yield)
  220 attempted, 126 validated (57% yield)
  230 attempted, 131 validated (57% yield)
  240 attempted, 136 validated (57% yi

## 6. Save the validated dataset
We write the kept pairs to `pairs.jsonl` — this is the file Step 3 (fine-tuning) will read.

In [ ]:
with open("pairs.jsonl", "w") as f:
    for pr in pairs:
        f.write(json.dumps(pr) + "\n")
print("saved", len(pairs), "validated pairs to pairs.jsonl")

if pairs:
    print("\n=== Example validated (Python -> Rust) pair ===")
    print("PYTHON:\n", pairs[0]["python"])
    print("\nRUST (compiles + passes tests):\n", pairs[0]["rust_solution"])

# Optional: To Google Drive
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copy("pairs.jsonl", "/content/drive/MyDrive/CodeGen_Group45/pairs.jsonl")


saved 194 validated pairs to pairs.jsonl

=== Example validated (Python -> Rust) pair ===
PYTHON:
 import math
def is_not_prime(n):
    result = False
    for i in range(2,int(math.sqrt(n)) + 1):
        if n % i == 0:
            result = True
    return result

RUST (compiles + passes tests):
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {
    let mut result = false;
    for i in 2..(n as f64).sqrt() as isize + 1 {
        if n % i == 0 {
            result = true;
        }
    }
    result}


KeyboardInterrupt: 

In [ ]:
for pr in pairs[:3]:
    print("PYTHON:\n", pr["python"])
    print("RUST:\n", pr["rust_solution"])
    print("="*50)

PYTHON:
 import math
def is_not_prime(n):
    result = False
    for i in range(2,int(math.sqrt(n)) + 1):
        if n % i == 0:
            result = True
    return result
RUST:
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {
    let mut result = false;
    for i in 2..(n as f64).sqrt() as isize + 1 {
        if n % i == 0 {
            result = true;
        }
    }
    result}
PYTHON:
 def square_nums(nums):
 square_nums = list(map(lambda x: x ** 2, nums))
 return square_nums
RUST:
 /// Write a function to find squares of individual elements in a vector.
fn square_nums(nums: Vec<isize>) -> Vec<isize> {
    let square_nums = nums.iter().map(|x| x * x).collect();
    square_nums}
PYTHON:
 def sort_matrix(M):
    result = sorted(M, key=sum)
    return result
RUST:
 /// Write a function to sort a given matrix in ascending order according to the sum of its rows.
fn sort_matrix(M: Vec<Vec<isize>>) -> Vec<Vec<isize>> {
    let mut result = M.

## What we built (and what's next)
- A **data engine**: Python problems → translated Rust → **validated by our harness** → kept.
- `pairs.jsonl`: our first batch of verified Python→Rust training examples.

**To scale up:** raise `limit` toward 354, and/or swap `TRANSLATOR` for a stronger model to lift
the yield. A few hundred validated pairs is plenty for a first fine-tune.

**Next — Step 3:** fine-tune codegen-350M-multi (LoRA) on `pairs.jsonl`, then re-run the Step-1
`run_benchmark` on HumanEval-Rust and watch the number rise above our ~1.3% baseline.

**Two things to Note:** (1) we train on MBPP and evaluate on HumanEval — disjoint,
no leakage; (2) the harness doubles as the data-quality filter, so even an imperfect translator
yields clean, execution-verified training data (the MultiPL-T recipe).